## pvdb2 generation
This notebook covers:
1. taking the output from the first pv database run in logan 1 to generate into a database
    * parsing
    * cleaning up

general flow:
1. got core gene orfs from the first logan search
2. filtered diamond .pro output for e < 10^-10
3. sto-sto orf calling
4. run against pfam PV models (this might be limiting, not sure)
5. filter hmmer output for e < 10^-6
6. select those orfs
7. annotated these orfs with source sra library metadata
8. wrangled headers to be consistent with previous input, eg, 9. papilloma.Late_protein_L1.Aneides_hardii_species:SRR6051504_11873_ka_f_2.929_35_661_2_REVERSE_SENSE_
9. concatenated this with the pvdb1
10. then clustered this combined db at 90% aa identity
11. unstick
12. send off to logan!

In [ ]:
#versions and stuff
zstd --version
#*** Zstandard CLI (64-bit) v1.5.5, by Yann Collet ***
awk --version
#GNU Awk 5.1.0, API: 3.0 (GNU MPFR 4.1.0-p9, GNU MP 6.2.0)
grep --version
#grep (GNU grep) 3.6
seqkit version
#seqkit v2.8.0
getorf -version
#EMBOSS:6.6.0.0
hmmsearch -h
# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.4 (Aug 2023); http://hmmer.org/
# Copyright (C) 2023 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
bedtools -h
# bedtools is a powerful toolset for genome arithmetic.

# Version:   v2.31.0
# About:     developed in the quinlanlab.org and by many contributors worldwide.

In [ ]:
%%bash
#get libraries from the first run, add sra metadata
grep ">" "/home/rnalab/js/logan_2/pv2_fasta_star_sto.fa" > pv_all_fasta_sto_sto_all_orf.txt
#parse and remove duplicates
sed -e 's/>//' -e 's/_.*//' "/home/rnalab/js/logan_3/pv_all_fasta_sto_sto_all_orf.txt" | sort -u > pvdb2_libs_sort.txt
#make grep airtight
sed -i.bak -e 's/^/"/' -e 's/$/"/' pvdb2_libs_sort.txt
#generate output file with library and associated host metadata in two columns, and demarcation of what level taxid it is
zstd -dc sra_taxid.csv.zst | grep -f pvdb2_libs_sort.txt | awk 'BEGIN {FS=",";OFS="\t"}; {print $1, $5, $6}' | sed -e 's/"//g' -e 's/\(.*\)[[:space:]]/\1_/' > sra_pvdb2_taxid.txt


In [ ]:
%%R
library(tidyverse)
##for new pvdb iteration
#from the last set of inputs, first analysis
pvdb2_sra <- read.table("sra_pvdb2_taxid.txt", sep = "\t")

#get the hmmer output file from the first iteration of logan runs, so those that are for sure orfs
#filter for e^-6
pv2_fasta_sto_sto_ieval_fil <- filter(pv2_fasta_sto_sto, eval_full < 0.000001)
#add column for library
pv2_fasta_sto_sto_ieval_fil$lib <- gsub("\\_.*","",pv2_fasta_sto_sto_ieval_fil$query_acc)

#join information from sra metadata
pv2_fasta_sto_sto_ieval_fil <- left_join(pv2_fasta_sto_sto_ieval_fil, pvdb2_sra, by = c("lib" = "V1"))

#define accessory genes, that don't need to stay in the dataset
accessory <- c("Papilloma_E5", "E6", "E7", "Pap_E4")
#remove these hits
pv2_fasta_sto_sto_ieval_fil <- filter(pv2_fasta_sto_sto_ieval_fil, !pfam %in% accessory)

#some are unknown, after joining, thats fine and annotate
pv2_fasta_sto_sto_ieval_fil <- pv2_fasta_sto_sto_ieval_fil %>%
  mutate(V2=replace_na(V2, "unk"))

#choose the max score annotation, e.g., what orf is this most likely to be?
pv2_fasta_sto_sto_ieval_fil <- pv2_fasta_sto_sto_ieval_fil %>% 
  group_by(query_acc) %>% 
  slice_max(score_one, n = 1) %>%
  ungroup()

#parse to use for header parsing
pv2_fasta_sto_sto_ieval_fil_key <- select(pv2_fasta_sto_sto_ieval_fil, query_acc, pfam, V2)
pv2_fasta_sto_sto_ieval_fil_key$info <- paste0(pv2_fasta_sto_sto_ieval_fil_key$pfam, ".",  pv2_fasta_sto_sto_ieval_fil_key$V2)
pv2_fasta_sto_sto_ieval_fil_key_final <- select(pv2_fasta_sto_sto_ieval_fil_key, query_acc, info)

#print list of contigs that are confident hmmer hits, as long as their annotations according to the sra metadata
write.table(pv2_fasta_sto_sto_ieval_fil_key_final, sep = " ", "hmmer_annot_pv2.txt", row.names = F, col.names = F, quote = F)

In [ ]:
%%bash
#have list of annotated ones, get list with contig information only
awk '{print $1}' hmmer_annot_pv2.txt > contig_list.txt
#this takes a while, giant .fa
seqkit fx2tab "/home/rnalab/js/logan_2/pv2_fasta_sto_sto.fa" | grep -f contig_list.txt | seqkit tab2fx > pv2_all_hits.fa

#map the annotation to the contig using this perl function
perl -lpe   'BEGIN { %id_to_function = map { /^(\S+)\s+(.*)/ } `cat hmmer_annot_pv2.txt`; } s{^>(\S+)(.*)}{>$1$2 $id_to_function{$1}};' pv2_all_hits.fa > pvdb2_orfs_info.fa

#reformat the header so that it conforms to the general input format of logan input databases
sed -e 's/>/> papilloma. /' pvdb2_orfs_info.fa | awk '{print $1, $2, $4, $3}' | sed -e 's/ //' -e 's/ //' -e 's/ /:/' -e 's/:$//' | cat masked.fa > pvdb2_v1.fa

#sort and cluster at 90% aa, for consistency sake
seqkit sort -l -r  pvdb2_v1.fa >  pvdb2_v1_sort.fa
usearch11.0.667_i86linux32 --cluster_smallmem pvdb2_v1_sort.fa -id 0.90 -centroids pvdb2_v1_sort_centroids.fa -uc pvdb2_v1_sort_clusters.uc

#copy centroid file here, to use as the first iteration of the database, which presumably has sequences that are intergration sites, etc
cp pvdb2_v1_sort_centroids.fa pvdb2_sticky.fa

#unstick
./logan_unsticker.sh -b "https://hgdownload.soe.ucsc.edu/goldenPath/hs1/bigZips/hs1.2bit" pvdb2_sticky.fa
./logan_unsticker.sh -b "https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/bigZips/sacCer3.2bit" pvdb2_sticky.fa
./logan_unsticker.sh "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.fna.gz" pvdb2_sticky.fa
#manually parse these outputs, to look for input contigs that have high confidence regions against those genomes (which they shouldnt, unless integration site)

#trimming rules are: if trimming, trim to either end, whichever is shorter. if > 50 aa remaining, keep. if < 50 aa remaining, throw out
#refer to pvdb2_trimming.xlsx for generation of bed file (Sheet 3) -> this is the culmination of mulitple rounds, see explanation

#mask based on the coordinates here, and replace masked sequences with nothing
bedtools maskfasta -fi pvdb2_sticky.fa -bed trim.bed -fo pvdb2_unsticky.fa -mc +
sed -i.bak 's/+//g' trimmed.fa.out

#reiterate with other genomes again
bedtools maskfasta -fi pvdb2_unsticky.fa -bed trim2.bed -fo pvdb2_unsticky_2.fa -mc +
sed -i.bak 's/+//g' pvdb2_unsticky_2.fa

#trim to only keep sequences those above 50aa
seqtk seq -L 50 pvdb2_unsticky_2.fa > pvdb2_unsticky_2_50aa.fa

#see minor additional trimming below

#final for real: pvdb2_90_final.fa

## pvdb2_trimming.xlsx and it's second version
These files serve as an overall record of things that were trimmed during the first two unstick steps. 

Additional sequences that were trimmed as a result of minor manual inspections and iterative runs recorded here: 

    * chr11_sliding:113681601-113682600	729	598	1000	-	papilloma.Late_protein_L1.Homo_sapiens_species:SRR1609909_37467_ka_f_27.187_L_1344185_L_168451_L_1350297_11_1_576_	9	52	192	97.7	4.11E-21	44M	RSPPRPSPGSLHYSDEDVTKYNDLIPAESSSLTEKPSEISDSQV	CGGGGTTCCCACGTGCTAGGTAAGTACTCATCCACTGAGCTGCTGAGACTGAACCCCGGGGTTCCCACGTGCTAGGTAAGTACTCACCACTGAGCTGCTGAGACTGAACCCCGGGGTTCCCACATGCTAGGTAAGTACTCATCCACTGAGCTGCTGAGACTGAACCCCGGGACTGCTGTATGCTCATCTATTGAGCTGCATCCCTAGCCCCTCTTTTAGTTACTCTTCAGCTCCAAGGTTCCTCCATAAGTCCGAGTTACAGAGAACCCGGGCAACGGCGGGGCTCTGGTGACTGTAGGGGTCCAGGCTCTCATGTGATTCCTGTGGCCCAGACCCCATTTGCTCTGGGGGAAACTGAGCAGGGCTTGGAGGTGGTGGTGAATGACCCATGAATGACTTCTGGATAGCTTTTCTCTCCCGCTTTGGTCTCCAAAGTCAAGGAAGGAAAGAAGGGAGAGGGATGGGGAGGAAGAGGAGGAGGAGGAGGAGGAGGCTGGCTGTGTGGTAGGATGCTGCTTATCCCTTCACCCACATTTGGTAGCAGCCACCACCTGTCTAGGCTTGTGCCCTCGTGCCCCTCGTGCCCCTTGCGCCCCTCACCTGGGAGTCAGAGATCTCCGAGGGCTTCTCAGTCAGGCTGCTGCTCTCAGCTGGGATGAGGTCGTTGTACTTGGTGACGTCTTCATCAGAGTAATGTAGGCTACCTGGACTGGGCCTGGGAGGGGACCTAGGGAGAGGAGTGAGGCTGAGAGGTAGTCTCAGAGAGGTCTGGAGGAACGCAGGCCACCGGCCCCTCCACCCTGGGTTCCTCCTTGTGACCACACTATGTCAGTCTGGGCCTTAGAGAAGTGGGTGGGGTGAGGGGGGACTTAGCAGAGGAGGAGAGAAAACAGAATACCACACTGAAGTCCTTGAACATATAGGGCAAATGGAATGTGTGTCTATGTGCCTGGGCACATACATGTATGCACAGACATTACTGAGTCCAGGTCAGCCTGAA	*	this seems to be a true positive…
    * chr6_sliding:134303201-134304200	368	496	1000	+	papilloma.PPV_E1_DBD.Homo_sapiens_species:SRR19350921_117961_ka_f_9.575_L_185896_64_1701_832_REVERSE_SENSE_	248	290	290	86	4.31E-19	43M	DDYSEVSTVADHCLNLKIRSLGQFINASPCLCLHEELEVVQNY	AACGTTTGGGGAGAGATCTGGTGGGAGGTGATTGGATCCTGGGGGGTGGATTTCCCCCACGCTGTTCTTGTGATAGTGAGTTCTCACGAGATCCGACGGTTTATAAGTGTGTGGCAATTTCCCCCTCACTCTCTCTCTCTTCTGCTCCACCATGGTAAGATGTGCTTGCATCCCCTTTGCCTTCCACCATAATTGTAAGTTCCTGAGGCCTCCCAGCCATGCTTCTTGTACAGCCTGTGAAACTGTGAGTTGATTAAAACCCTTTTCTTTATAAATTACCCAGTCACAGGTAGTTCTTTATAGCAGTGTGAGAACAGTATAATAGTTATATATACTTATATCAATAACACTTAAGTATTCATGCAGGGATGACTACAGTGAAGTAAGTACAGTTGCAGATCACTGTCTAAACTTAAAAATCAGGAGTTTGGGGCAGTTCATAAATGCAAGTCCCTGTCTATGCTTGCATGAAGAATTGGAGGTTGTGCAAAACTATTGATACAAATGTTAGCTTCTGGTCGGTGTGCTTGGCCATATGGCTCTGCAGATTCTCATCAAGTTCGATAGCTCTAGGCAGGTATGTGAACAACCTCTAAGAATGCCCACCTCAATTGTGTCTGCTAGTCAGCTAAAGGCAGAATGTAAGGATTTTGCTGTTATAATGAATGATAAGTACATTTGCTAATGTTTACTAAAAAGAGTAATAATATTAACTAACAGGTAGTGTTACTTATCAGACACTGGGTAAGAGCTATTATTATTCACATTTTAGAGAGGCACTAACTTCAGATGCAAATAAGTTAATTTCCTCAAAGTCACAAGTGGAAGCAGTGTGGTTTCAAAGTCTAAGTTCTTAACCAATGTGGCGATATATAGCTCACAGCTGTAGCTATTATTTTTCCTGCCTCAGTGCACATGAATCATCTGCCTATGTGTGAGTGGCAGAAATGAGTGAATTCCTACTACAGTTACCATGCAAGCAGGGGCAGGAAACATGGGA	*	cleaned, 248-290 removed 
    * chr5_sliding:181752001-181753000	149	223	1000	+	papilloma.Late_protein_L1.Homo_sapiens_species:SRR5008130_66473_ka_f_3.303_4_3_185_	1	25	61	100	4.05E-10	25M	RGSRSRLWKGQGRVSVRPSCRESRS	CGCGCGCGGGCGGGAGCTGTCGGGGCGGGGCGGCACCGCCGAGCTCCGAGCTGTGAGGACCGCCCCGGGCCTTGGGCGGGGCGGGGGCCAGGGAAGGGCGTGCGCAGGATTGGGGAGCACGTTTTGCGTCTTGCTGTGCGTGAGGAAGCGGGGATCCAGGTCAAGGCTATGGAAAGGTCAGGGAAGGGTGTCAGTGCGCCCGTCCTGCAGGGAGTCCAGATCCGAGAGGTTAGGGCGGGACCTGGGGCGAGCTGGGCGGTGGGGGAACAGTGGGGTCATCAGGAGGTGGAGGCCCAGCGGGAGTGTGGCGGGGGAGGCGGGGAATCAGGAGGGCCAGGTGTTTGGAGGCGCTGGGGTGGGAAGGGTCCAGGCGAGGTTGCGGGTGGGTGCCAGCCCCGTCTCACATTTGGATGAAGTAGGAAAAGGCAGCTTTTAGGGTACACCCTGACATGTTCTGTCAGGGCCCTTAAAGGACCCACGGGCCGGAGGAGAGGGGAAAAGGATGAGGGCTCAGCCTGGCAAAGGACAGAAAGTGGTGGGTGTAGGAGGTGGACCTTGGTCTGGTGGCCTAGCCTGCCCTGTGCTCCCTGCAATGCCGGGTGACTTGGGTCACTGGGCGGCCAGGCCAGAGGTGGGCCTCTAGGTGCCCCCCATTCTGGATGCTGAGGGTCCCTGTGAGGGTCTGGCGGCAGAGGCCTGCGCTTATGGCTCAGCAGCTAGGACTGATGGGTTGAATAAGAAGGGCCCAAGTTAGGCCAGTTGTTCTCACAGACCCAGAGGTCCCTAGAGTTTATGATGCCCAGAGTCTAGGAGAGGGAAATGCACACACATGCCCTCGAGGACACCAGCATCCTGACATCCAAGTTACTGTACTGCCCTGGTTAGCAAGAAAGAACATCTGTTCCCTTCTCTATGCAATTGTTACAGTAACTGTTGTTACATTTCTTAAGATACATAGCTATATATGTAGTCACTCTGCAAACACACACACTCTGATAGA	*	entire sequence removed
    * chr6_sliding:20899201-20900200	862	939	1000	+	papilloma.PPV_E2_N.Homo_sapiens_species:SRR21903069_67109_ka_f_10.121_L_290144_L_294868_L_295374_60_279_1_REVERSE_SENSE_	1	26	93	96.2	8.92E-10	26M	GGAHLARGAKENQRGRHKGGLNTGTE	CCCAGCCATAACCCTACCACCAGCACCAACAGCCACCCCCGCCCTACCTCCAGATTGCTAAAAATCCTGGCTGAAGGGGTTGTAGTGTTTAATCTCCACAGAGCCCTCTTTGCTGTTCATCAGTCTATTTTCCTTTTCAGTACATTCTTCCATATTCATATTAGCCATATCAAATCTTCTTCGTCTACTTCAAGTATAATACTCCAACTTTACCTAACTTATTCTCAGGAACAGGATTGGACCTCACAAGTACCACCTCAGCTCTCCTCCTTATCAGCCCCAAAATGCAGTTATATACTGGCCATTGTCTTCATTCTTCCTGCCTAAGAAGTTGTTTCTCCTGTCTTCAGTGGATTGAAGTAGATTCTAATGGACAGTGCTATGGTTTAAATGTTCCCTTCAAAATTCATGTTGAAATTTAATTGCCATTTTGATGACATGAAGAGATAGATTTAAGTGATCACCTCTTAAAGCCCTCGTGAATGGATTAATGCTGTTATCACCAGAATGGGTTAGTTATCATGGGAGCTCAGCTCCCTTTTCCTGTCTCAAGTGCTCACTTGTCCTTCTGCCCTCCACCACAGGATGATACAGTATGAAAGCCCTCACAGATGTGAACGCCATGTTTTGGACTTCCCAGTCACCAGAACTAAGAGCCAAATAAACTTCTGTTGTTTATAAATTCCCCAGTCTATGGTAATCTGTTACAGCAGCAGGAAACAGGCTAATATAAGTGGGGAGGTAAGACCAATGAGAGTAACAGATCTGGAGAACTGAGTGTAGATCATGTTCTGGAAAACAGAATCATTTGGGATAAGAGCCTGTGCAGAGTCAGCCAAAAGCTTTGGCCCAGAAATCTGAGGTGGAGCTCACTTGGCCCGTGGTGCTAAGGAGAACCAGAGAGGAAGACATAAGGGTGGACTAAACACAGGAACTGAGGAGGAGTCAGAGAATCCAAGAACAAGGCAGGTGGGCCTAGGTACTAGAGAATCTGAAGAAA	*	cleaned, 1-26 removed
    * chr6_sliding:20900001-20901000	62	139	1000	+	papilloma.PPV_E2_N.Homo_sapiens_species:SRR21903069_67109_ka_f_10.121_L_290144_L_294868_L_295374_60_279_1_REVERSE_SENSE_	1	26	93	96.2	8.92E-10	26M	GGAHLARGAKENQRGRHKGGLNTGTE	AGAATCATTTGGGATAAGAGCCTGTGCAGAGTCAGCCAAAAGCTTTGGCCCAGAAATCTGAGGTGGAGCTCACTTGGCCCGTGGTGCTAAGGAGAACCAGAGAGGAAGACATAAGGGTGGACTAAACACAGGAACTGAGGAGGAGTCAGAGAATCCAAGAACAAGGCAGGTGGGCCTAGGTACTAGAGAATCTGAAGAAACCTGGTGTATTAGGGTCATCTGGGCTGAATGGAGTCACAGTGATGAATGGAAGCCCAGGGAGTCTGTCAATACTTTGTCTGGAATGTTGAAAGAAATTCCTAATTGGCTACTTTTTTCTAACTCTGGCTCACCTCCTGCATTTTCAACAATAGTAATAGTTAACATTTAAGGGTTTTCTCTGTGCCAAGCAGTATCTTAGAACCATATAGACATTATCTCATTTGATCTTGACAACATTCAGCTAGTGTATTTATTAGTCTGCTCACGCTACCATAACAAAATGCCATAGTCTGAGTGGCTTAAACAATAGAAATATATTTTCTCACCATTTTAGAGGCTGCAAGTCCCAGATCAGGTGCCAGCAGATTCTGTTTCTGGTCAGGTCTCTGTTCCTGGTTTGCAGATGGCTGCCTTCTTCCTGTGTCCTCCACTGGGCCTTTCCTCTGTGTGTGACATGGGGAGAGAGAGCTCTGGTGTCTCTTCCTTTCCCTATAAGGACACCAATTCTATCAGCTCAGGGCTCCACCCTTAAGACCTCATTTAACCTTAGTTACCTAAAGGTGCTATCTCCAAATACAGTCACACTGGGGTATTAGGGCTTCAACATGTAAATCTTAGGGGGACACAATCCATTCATAACAGCTAGTTAGGTATTATCCTTAATTTGTAGATTAAAAAATAGAGACTTAAAGTGGCTCAAAGGCACATAACTAGTTACCAAATGCTGCTCGACCTTATAACCTGAATGTTCAATTGGTACTTCAAATTCAGATTGTCTAAGATGGAAATCATATTCCCT	*	cleaned

recorded in pvdb2_trimming_2.xlsx

also removed:
    * papilloma.PPV_E2_N.Homo_sapiens_species:SRR1611060_1514_ka_f_1341.901_L_106420_L_107106_4_1_180_
    * papilloma.Late_protein_L2.metagenome_species:SRR13789600_144930_ka_f_2.419_15_223_14_REVERSE_SENSE_

for some borderline confidence hits

another manual iterative clean

    * NC_000071.7_sliding:104668801-104669800 865 981 1000    +   papilloma.PPV_E1_DBD.Homo_sapiens_species:SRR2984640_3511_ka_f_122.215_L_33155_60_1140_1_REVERSE_SENSE_ 274 313 313 80  4.61E-11    10M1D29M    RTDITEPEREKRRKVDSHPSPSHSSTIKDSLVKLKESSA TCGGTACATTGCCTCGAAACAGGTGAATATACACATATCAGGAATATCTTGATTGTACTAACAAAAATACTACCATGGTACCCCAAGGTTTTAAATCTGGGTCAGGCTTTGGAAAGAAGAGTGCATAAAATCTGCCAAGAAGAAAAAGAAAAGAAACCAGATCTATATGCATTAGCAATGGTCTACTCTGGGCAGCTGAAAAGTAGAAAGTCATACATGATACCTGAAAATGAATTTCATCACAAAGACCCTCCTCCAAGAAATACTGCTACCAATCTACAACCAAGTGGGCCTTGTAGTGGGCTGCCTTCATCTATAGGAAGTATGTGTAAATTAGATGAGAGCAGTGCTGAGGAGGCCGACAAATCACGAGAAAGAGCTCAGTGTGCTGTGAAAGCTGCTAATAAAGCTTCCAGTGTCACACCAAAAGGGAATTTAAGCAATGGAAACAGTGGCTCTAACAGCAAAGCTGTTAAGGAAAATGACAAAGAAAAGGGCAAAGAGAAAGAGAAAGAAAAAAAAGAAAAGACCCCAGCTGTTACTCCAGAGGCCAGGGTACTTGGTAAAGACAGTAAAGAAAAACCCAAGGAAGAACAACCAAATAAAGATGAAAAAATAAGAGAAGCCAAAGAAAGAATGCCTAAATCTGATAAAGACAAGGAAAAATTAAAGAAGGAAGAAAAAGCTAAAGATGAGAAATTCAGGATCATTGTTGCCAATGTAGAATCAAAATCCACTCAAGAAAGGGAAAAAGAGAAAGAGCCCTCAAAAGAAAGAGATTTAGCAAAGGAAATGAAGTCAAAAGAGAATGTTAAAGGAGGGGAAAAAGCACCAGTTTCTGGCTCCTTGAAATCACCTATTTCCCGAACAGATATCACAGAACCTGAAAGAGAAAAACGTCGCAAAGTTGATTCCCATCCTTCTCCATCACACTCTTCCACAATAAAGGACAGTCTTGTCAAACTTAAGGAATCTTCAGCAAAGCTCTATATCAACCATA    * cleaned, 274-313 removed
    * NC_000071.7_sliding:104669601-104670600 65  181 1000    +   papilloma.PPV_E1_DBD.Homo_sapiens_species:SRR2984640_3511_ka_f_122.215_L_33155_60_1140_1_REVERSE_SENSE_ 274 313 313 80  8.30E-11    10M1D29M    RTDITEPEREKRRKVDSHPSPSHSSTIKDSLVKLKESSA AAAAGAGAATGTTAAAGGAGGGGAAAAAGCACCAGTTTCTGGCTCCTTGAAATCACCTATTTCCCGAACAGATATCACAGAACCTGAAAGAGAAAAACGTCGCAAAGTTGATTCCCATCCTTCTCCATCACACTCTTCCACAATAAAGGACAGTCTTGTCAAACTTAAGGAATCTTCAGCAAAGCTCTATATCAACCATATTCCACCACTACTGTGCAAGAGTAAAGAGAGAGAAGCAGACAAGAAAGATTTGGACAAGTCAAGGGAAAGATCCAGAGAAAGAGAGAAAAAAGAGGAAAAGGACAGGAAAGAGCGGAAAAGAGATTATTCAAACAATGACCGAGAGGCACCCTTGGACTTAATCAAGAGGCGGAAAGATGAAAATGGAATATTGGGGGTTTCAAAACACAAAAGTGAAAGTCCCTGTGAGTCTCTTTATCCAAATGAGAAAGACAAGGAAAAAATGAAGTCAAAATCCTCAGGCAAAGAAAAAGGTGATTCATTTAAACCTGAAAAGATAGATAAAATATCCAGTGGGAAAAAGGAGTCCGGACATGATAAGGAAAAGATAGAGAAGAAAGAGAAATGGGATGGTTCTGGAGATAAGGAAGAGAAGAAACATCATAAAACCTCAGACAAGCACAGATAATGAAGACTCAGCCAAAAGTGTGGGCAGGCCTCTGAGCTGAAGGTCTCCCCTGCTGAGGATGCCAGAGCTCTTGGTGTCACTCTCTAAACAGTATCTCTCTTAATTGAGAGCCTCTGCTGCTGTCCATCTTATGAAAATCCTTTAAGTCAGAAGATGAATATACATCGCTATCCTCTCGAGACATTGCAAAGTCACTGGCTTTCAGAACATTATTTTCACCTCTGGGCAGTTCCTTGCAGCAAGATCTCTGTGCGTACAATTTTTTCTTGAAATAATACCACCTAGAAACAGCATCTATGAAATTTCCACCAGCTGAGTCTCAGATCACCCTTCTTAGAAAGGTGAGTTAAT    * cleaned, 274-313 removed

    * NC_000018.10_sliding:1505601-1506600    541 419 1000    -   papilloma.PPV_E1_N.Human_papillomavirus_type_16:HE984555    52  92  92  100 1.52E-25    41M QRFTFYCGFTPHAVWQKALFHTPFQKSSLAEGSPPITCWSG   GTCTACAGGTTTCTGGACCTAGAGCTGCTGGAGTCACATTTGGGTTTAACATAGAGATGACTGCACATCACCCAGGGACCCTGGACTTGAAGTTGAGTTTGTGACTGGATGGAATTTTTTTGGGTCCTCCTTTGGGAAGCTATTAAATGTGAGCCATATATTGGGAGAAAATAACAAAAGGATATTTGGTGACTAGAAGGATAGACAGGGCCATAGACAGTAATGCTCCCCCAAATTTCTATATTTCCCCGAATTCCCACTCATGTGACTGGTCATGGCCAGTGAACTCTGAGTGGGAGCAATGTATCACCTCCATGACTCAATTTATCATTGAATGTCTCACTCTGAGATCATGAAAAGCCCAGGTGTGGTCTGCAGCCTCTCTCTCCCTGGTGTGCAGACCAAACATGCCTTTTATTCCAGACCAACAAGTAATAGGAGGAGAGCCCTCTGCCAGGCTTGATTTCTGAAAGGGAGTGTGAAACAGTGCCTTCTGCCAAACTGCATGGGGTGTGAACCCACAGTAAAAAGTAAACCTTTGTTATGTTAAGCGTCTGAGGTCTGGGCTGGGAGGAGTGTTTTTTTTCCCTAAAACAAGGATCTAGCTTATCAAGATGAAATCAGTTATGTTTGGTAAAGCTGAAAATTCTGTATAAAGCATATCTTTTTAAATGTTATTTCATTTTCCTACATATTGCTGAGGACACTTCATGTAGCTTATCTATCCTATTCACCCATCTCTTTGCTCTTGAGGAAAAATAAGTATTAAAAGAACAATGGTGAAGAAGCAGGCACAAAGCTCTACCTACCCACAAACGCGCTGTTTCAAATGCAACAAACTGTGGGTAGAAGACTTTGATTTTCCCTTTTCACATAATGGTGCATCAGAGGTTAGAAACACTCAGGAGAAGGTTACTGAGAATATCCTTTGGGAATTCCATAATGGGTGCTTGCTATTTGTGCATTTCTTCTAATCAATGCAAAATAGCAGATTGTGGGT cleaned, 52-92 removed
    * NC_000008.11_sliding:127852801-127853800    595 473 1000    -   papilloma.Late_protein_L1.Homo_sapiens_species:SRR18511782_219413_ka_f_9.307_L_2566002_34_399_1_REVERSE_SENSE_  1   41  133 100 1.33E-22    41M RGPSNSLVGGLHCCQPLGQPEPPPAESPSAHSISSLKILCC   CTGGACACACGTTATGACATGTTCTGATGATCTGGCTTAGACAGTGGGGCCCTCGAGGTAGGCCCAGAGGACTTGGTCCTCACTGCCTCTGTGGCGCCTTGCACTGGGTCCAGCTGACGTGGAGAGAGACTCAGGAAACAGTGGCTGAGTGTGACTTTGGCTGGCATAGTGGTTGCTGAGAGAACAGACAAGGTTCTCTCTCACGACATACAGATTTCAGATCAGGGAAAGTCCCAGCTGGCATAAGTTTATCGAGCATCTCCCATGGACAAGATCAGCTGTGGGTGGAGCCTTGAAGTACATGGTAGAAGGACAGCGAGTCTTCCCAGGCCAGGGCTTCAAGTGAGGAGACAAGATATAGCCTCCCAGAGAATTCCTATAATGCAATCGTGAAAGAACCATACCCAGCAGGAGGCCGGGGAAAGTGACTCCTGCAACTCTAGGAAGGCTTCCTGGAAGAGGTGGAACGTGAGCAGCATAGGATTTTGAGAGAAGAAATGGAATGGGCTGAGGGAGATTCTGCTGGTGGAGGTTCAGGTTGACCTAAGGGCTGGCAGCAGTGGAGGCCCCCCACGAGTGAGTTTGAGGGGCCTCTTTAGCTCAGTCCAGTTGAGGCAGCAGAGCCTTTCCATAGGGGTGTGGTGTGACCTGAATGTTGGGCACGTGGTCGTAACTGAGCTTTAAAAGTGAATGAGAGGAGCCATGCGTGATGGCTCGAGCCTGTAATCCCAGCACTTTGGGAGATCAAAGCTGGGGGATCACCTGAGGTCAGGAGTTCGAGACCAACCTGGGCAACATGGTGAAACCCTGTCTGTACTAAAAATACAAAAATCAGTTGGGTGTGGTGGTGGGTGCCTGTAATCCCAGCTACTCAGGAGGCTGAGGCAGGAGAATCGCTCCAACCTGGGAGGCAGAGACTGTAATGAGCCAAGATTGTGCTGCTCTACTCTAGCCTGTCTCAAAACAAAAAACAAGAAACAAAAACAAAACAAAACAAAAA cleaned, 1-41 removed
    * NC_000008.11_sliding:127220801-127221800    720 607 1000    -   papilloma.Late_protein_L1.Homo_sapiens_species:SRR9609700_3881_ka_f_22.860_23_459_127_REVERSE_SENSE_    74  111 111 100 7.39E-21    38M STLKMQDRENHHLLNGIYTAGFVPGTHSTVPSYPLLHP  CCAGTATCACAAATCTAGAATGACAAGCCAAAGCCAAGATATAGTTGGCCAGGGGCAGCCAGTATCCTTCATTTTCTTATTCATTCATTAATTCACTCATTTATCAGGTGGACATTTGCAGGGCACCAGTGATGCTGTCAGGCACCATGTTTGGTGATGAGTTACCAGATGGTCCTAGCAGATGAAAAGACTGGGAACCCACTGGGAAGGATACAAAAATTATTACACAGCTATCAGAGCAAGAGGGAGGTTAGTAAAAGCTGGTGGACCTTAAAGTTTCTCTACTTTTGCAAGTGTAAAAACTGGGGTAAAGATAGAGTTTGGGATAACGGACACAGCCATAGCCAAAGAATGAGTCTAGTGTCTATAGAGAAGTTGTGATATTTGGAATTTTCCATGTGCAAACACACACGCACATGCACGCGCACACGCACACACACCACACACTCCTCACCTTTGCCATTTTCATAGGGAAACATGAAGAAACAAGTATATGGATTTTGGAATTGGGCAGTGTTTTATTTCAGCCCCATGTTCGCTTGCTAGCTATGTGATATGTGACTTTGGGCAACTTGTTTATCTTTTCTGGGAAAACATATACCTCTAGGGGTGCAGTAAGGGATAACTGGGAACAGTGGAGTGTGTGCCTGGTACAAAGCCTGCGGTGTAGATGCCATTTAATAAATGATGGTTTTCTCTATCCTGCATCTTGAGAGTAGATCCAATCTATTGTTTATTCTTTACAGTAGGCATGGTAGAGTAAAAGGAACAAAGGAATCGAGGGAGGAAGGGAAGAAAAAATGAGAAAAACCATAAGGCCAGGCGCGGTAGCTCACGCCAGTAATCCTAATACTTTGGGAAGCTGAGGCGGGTGGGCGGACCACGAAGTCAGGAGTTCGAGACCAGCCTGACCAATATGGCAAAACCCCATCTCTACTAAAAATCCCAAAAAAAAAAAAAAAAAAAAAAAAAAGTTAGCCGGGTATGGTGGCACGTGCCT cleaned, 74-111 removed
    * NC_000008.11_sliding:127249601-127250600    951 826 1000    -   papilloma.PPV_E1_C.Homo_sapiens_species:SRR5605601_73310_ka_f_5.553_20_3_791_   222 263 263 81  1.87E-15    42M HIQNLPFGQSGNREQRTRMNWAGKPFTKIRKEERGETPSSPP  TGATCATGTGCCAGTAACTTCCAGGTTTTTTTTTTTTTTTTTGATCATTTAATTTGAGTTCATCAGAAAGAGTCCATTTTCCTCCTGTTGGGAATGTTTTTGATCTACATGTAAAGTCACTTGGTCATGTCTCTTGCCAGGTGATCATTTTCCCTAGGTCAGTGGTCATCATTATTAAAACGGACATTGATTCCAACCCCTGGGGTTTCTGATTTAGTGGTTCAAGGATGCGGCCCCAAAATTTGCATTTTTAAAAAAGTTCCCAGATGACGTTGGTGTTGATGTTGCTCATCCCAGACAACACTTTGACTCCCAGAAATCAAATTTCTCTGACTCTGTACTTAGAGAAAAGCATTTCTAGTTATGTCCAGTCATTGATGTCTCTATTTCCTGTGTGTGGTTGATGACTAGGTATGTGGAAAGAACCCTATATACAGGACTGTTGTGCTAAGGTCATTATCGTAATATTGATAATTATTCTTTGTATCCCATTACAAGTTGCAAGGTATCTTCCCATACTCTATTTAATGTAACTTTTGAACCAGTGATAAGTTGGTTACTGTTGCTCCATTTTACCCAGGAAACTAGTTCAGAGGTGTAGTGTACCTTGAGACACATCATTGTGAGCTCATGGTAGAAGCTGGGTGGCAAATTCTCACCTTTTTATTCTAGAGCCTGTGTAATCTGCCACCCAGCAGCTATTTTATAGCTTTTGAACCTAAATTTTGATGAGAACTATCTCGGTGGTGCTCTGAAAATCACTGCAAATTAAGACAACACACAGGTTTGTTTCATAGTGTATTTGTGCTACCAGCTGCATCATCAAGGTGGTGACGATGGTGTTTCCCCTCGCTCTTCTTTTCTAATCTTAGTAAAGGGCTTGCCAGCCCAGTTCATCCGAGTTCTTTGCTCCCTGTTCCCTGATTGTCCGAATGGCAAGTTTTGTATATGCTAAGGTTCACTTCCTAACACGAATTCTCTGAGGTTCATTGGTTAATGA cleaned, 222-263 removed
    * NC_000008.11_sliding:127250401-127251400    151 26  1000    -   papilloma.PPV_E1_C.Homo_sapiens_species:SRR5605601_73310_ka_f_5.553_20_3_791_   222 263 263 81  1.87E-15    42M HIQNLPFGQSGNREQRTRMNWAGKPFTKIRKEERGETPSSPP  ATTTGTGCTACCAGCTGCATCATCAAGGTGGTGACGATGGTGTTTCCCCTCGCTCTTCTTTTCTAATCTTAGTAAAGGGCTTGCCAGCCCAGTTCATCCGAGTTCTTTGCTCCCTGTTCCCTGATTGTCCGAATGGCAAGTTTTGTATATGCTAAGGTTCACTTCCTAACACGAATTCTCTGAGGTTCATTGGTTAATGACCCTTTCTTCTCAGGAACCAGAGAAATAAACATGATGTGGGTTGTGTTCTGTGCAGGTTGCTTCAACTTTTCTCCTCCATACTTTCTCTCCGTAGCATCACACATCTGATGAGTGGGAAAGCTGAGATTTGAACCTGTGCCATCTGTTTCAAAGCCTGCCTCTAACCATTTCACATGGCCCTGCTCACTGCCCTCAACAGCACTTCCTGTTCTATCAGAATCTTGCTGAACTCCAATCTGACATTGCGGGTGCTCAAGAAATTCACCCCACACATTTGCATCTTCACTGTGCTCCACCCAGGCTGGCTGTTCCTGGTGGAGGAGTTTCCAGCTCACCTGGGGTTCCTACCCTCATGCAGGTGACAGGTGTCTGAGTAGCTGAAAACACAAGCTTTAGACCCAGGCAACTCTAAATATGACGCACCACTTTTTCTTCTTACTGTATAACCTTGAGTAAGTCACTGATTTTCAGAACCACTGTTTCTTTATCTGTAAAATGGGTACAAGAATATCAAGTGTGCGTAGTTGCTGTGAGAATTAAACGGATAATTTATGCAGCAAGAAGCAGCTCAATCATTGGAGCTTTTATTATTTCTGTCATCCTATTGCCTCTGCCTAGAGCCCCTAATCTACTCCTTTTCTCTCTCTCTCTCTCTCTATCTCTCGACCAGGCTGGTCTTGAACTCTTGAACTCTTGCCCTCAAGCAATCCTCCCTTCTCAGTTTCACAAAGCATTGGGATTACAGGCATGAGCCACTGCAACCAGCCCAAGCTTGATATTTTTATACCCATTTTACAAA cleaned, 222-263 removed
    * NC_000010.11_sliding:62400001-62401000  608 694 1000    +   papilloma.Late_protein_L1.Homo_sapiens_species:SRR902917_87947_ka_f_9.668_28_253_2_REVERSE_SENSE_   1   29  84  100 1.78E-12    29M VVGMTVDCTLGASCIVTILEALRVSPSPF   AAATTACAGAAAGAATAAAAAAATTAAATCAATCTTAAAGCTCTAACTTCTAAAGTAACTGGCCCAGTCTCCAGTAATCTTGGCATCTGGGCTTCTATGCAGTGGTCACTAATTTTCAGGACCAAGGCCACACAAAATGTCAGGCCTAGGGAGAAAATTCATCTGTGCCCACTGCTCTCCAAAGTGCGAGGCAAGGAATGGCAGTGTAAAGTTCTGTTAATAGCAGTAAAATGAAAATATTTGTGTTTGTGTATAACCTTCCCCTCAGCTAATTTGAAAGCCTCCAAAATAAGGATTCCCATTCCCCGAGTATTCTGGTTAATCAAGATTTCAATTTCTGGGTTGCTCAAGGGACTCGTTCAGTCAGACTTCAGTTCTCATTCCGACAGGGTGTCTTTCAGTTCGTTTGTTTGATTGAGGTTTTTTGGTGGCCTAGATGCATGTTTATTCAGATTTTGGTACACCTCTGCCGTCTTCTTTGGCTGAGTATTCTGCACCCACAGACCATGCTGCCAGCCTCTATCTTAATAGCTGCTTCTGTGGATATGGCTGGGGAGCGAAGTAAAATCCTCTTTTGATTAGCACCCAGCATGGGCTGGGTGGCTAGGTGGTTGGAATGACAGTGGACTGCACGCTTGGTGCATCGTGCATCGTGACCATCCTGGAAGCACTGCGGGTGTCGCCAAGCCCTTTCCTAAGACCTGCTTCCCGGCCAGTGTCAGTGGCCCTCTCTCTCTCTGCTATGGGCATGACTTTCTCTTCGCTGGCTTAACTTTTCCACTGGGTGGTTCACTTTGCCTCCTGCTGCTTCTGTGCCCTGTTAAGGGCTTCAGGTATCTTTCAACCAAGTATCTGGAGTGTTCACTCTATGTTGCATTCTAAAGTAATTTCTGAAATAAACAATGCATGTAGGAGCCAGAATCTGACACTGTCTTCCCCTCCTTCCCTCCCTAAAATGGCTTTAGTTTCCACAAAGAGCTGTTAAGAATTTCCTGCTGTA cleaned, 1-29 removed
    * NC_000008.11_sliding:38304001-38305000  594 680 1000    +   papilloma.Late_protein_L1.human_vaginal_metagenome_species:SRR21663145_179428_ka_f_16.546_11_39_581_    153 181 181 100 2.29E-12    29M TNETKAYIYSRRITAPFGMITTDENVGGY   ATCAATCTCTTGGTGTACTTAAACGTCAACAGCTGGCAACAAGAAAGCCCACATGGCTGGCTCTGAACTAACACCACCACATATGCTGCCTTGGCTGTGTCCCATCAGGCCCTCCTCAATGCCAGACCAAAACATTATGATAAAATACAGCATAAACAACTGAAAAATGGATGTGCTAGTTTTGCCTTCCAAGTGAAATCTCAATCAGTTGAAAGTTCTGGTTTAAGATTATGAAATGGGAAGGATCTACTTTGAATAGAAGAGTTGTTTGCAGAAGGGAAAAGATTAATCTTTTAATGCTCCAAACAGAAAAACTAAGAACTTAACTATAATTCTGTTATTTTTAGAAATGCAGATTTTAGCTTAACTTATGGAAGAATTAGTATTCTAACACTTTGGACATCTAGCATTATAAAAGTTACTGGAAATGTTCATCTCTCTCAGGGATATATTACAGTGGAATTCTGCACTGGGTAGGAGATTCTACTAGATGGCCTGAGTTCTTTCCCAATGCTGAGACTCTATGTTAATGGAGAAATATGCCAATGACCTCAAAAATAAATGAAGAGAAAAGTCAGACACTCACCTCTGGCACAAACGAAACAAAAGCCTACATTTACAGCAGAAGAATTACTGCTCCGTTTGGAATGATTACTACAGATGAGAATGTAGGAGGATACTAACATGCTTCCGGCCGCAATGCAAGCATCTCCAGAGTGATAGGCAACTGGACATCTTAAACATCTCATCATGCGGCCTGTTTAAAGACAAGTCAAAAAGGAATCTAATAAGGCATCAGCGGGACATAAAAATAAGTTTCTGGAGTTGGGCTTTTTTGGGTCCCTCTGAGATGCTAGTTACAGTAGATGGAAAGCATTTTTTTCTTCCCTAGTCTGGCTCTGTCATCCTATGCTATATGGCCCCTAAATGTAAGCCTAATAGTCACGTAGAGATCATCCAAAGAAAAAATTATTTGTTCCAATGCTGCAGAGACGGTGGT cleaned, 153-181 removed
    * NC_000011.10_sliding:59574401-59575400  187 122 1000    -   papilloma.Late_protein_L1.Homo_sapiens_species:SRR5337859_53577_ka_f_45.039_L_1108801_L_1108802_4_1_222_    1   22  74  100 9.46E-09    22M FTICTTFLADPNQIPANGRSLE  CTTTTAAAAAATTATTTAATTTAGTAGTTTTTTTTTTTTGGAAAAAAATCAATAAGGCAAAAAAACTGAAAATATATATAGTTGTGTTTTGTTTTCATGTGACTTTATTCCACCTGAAAGATTCTAAACTTCTTCCATTAGCGGGAATCTGATTGGGATCAGCCAGAAATGTAGTGCAGATGGTGAAGGAAGAGAAGGGGATAAAGCAGAGAAGTCCAATTTAGTTGGGGAATTCTCACTCTATCCAAAGCCCCGGATGAGGTCACTGCTTTTATGAGCCTCCTTGAAAGAAGCCTTGACATAATCCCTGGCTTATTTTGGGAATAAGGTTCTTAGGAGGAAATTCCCCAGCAGTCAAAGTCACTCCTCTCCCATGTTTATCACAACTAGGAAGGAAGGGATAGGGGAAGAAATAAAAGTATTAATTTACCCAGTAAAACCAATTTCCCCATAGGGATGGCTTCCTTTGGAATGGTATTTCCCCCAGGACCCACGCTGGCTGAGGCTCAGCAGTTAAGGAACAAAAACAAAAAATACTAACAAAAAGCATACAAGGAAGGCACCTCAATTTGTGATCCTCAACCAAGGGTGGGTTGCACAGGGGATATTTCTCAACAAGACAACAAAAACAAATACTGCACAAACAGAGAAGGAAGCCAGGGCCCCACAGGAGCAATTATCTGATCCACCCCACATGACGGTGCTTTAAGCCCCACATCCTAGGAAGAAAGTTCTTTTGGAATTCAGCCTATGCGCCGGACAGAGCAGAATTAAATTGGAAGTTGCCCTCCGGACTTTCTACCCACACTCTTCCTGAAAAGAGAAAGAAAAGAGGCAGGAAAGAGGTTAGGATTTCATTTTCAAGAGTCAGCTAATTAGGAGAGCAGAGTTTAGACAGCAGTAGGCACCCCATGATACAAACCATGGACAAAGTCCCTGTTTAGTAACTGCCAGACATGATCCTGCTCAGGTTTTGAAATCTCTCTGCCCATAAAAGATG cleaned, 1-22 removed
